# Handwriting branch: input → preprocessing → kinematic features

Runs `data_input` → `preprocessing` → `features` for every handwriting recording.
Output: `kinematic_features/output/kinematic_features.csv` (not committed).

In [2]:
import sys, warnings
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "data_input").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "data_input"), str(ROOT / "preprocessing"), str(ROOT / "kinematic_features")]

from load_data import load_manifests, load_signal, route_by_modality
from preprocess import preprocess
from features import extract

META = ["sample_id", "source_dataset", "task", "label", "subject_id", "subject_group"]
warnings.filterwarnings("ignore", category=RuntimeWarning)  # empty-slice warnings on missing channels

## 1. Load manifests and route by modality

In [3]:
signals, images = load_manifests()
handwriting, _ = route_by_modality(signals, images)
print(len(handwriting), "handwriting recordings")

2385 handwriting recordings


## 2. Preprocess and extract features

In [4]:
rows, skipped = [], 0
for _, r in handwriting.iterrows():
    df = preprocess(load_signal(r["path"]))
    if df is None:
        skipped += 1
        continue
    rows.append({**{k: r[k] for k in META}, **extract(df)})
out = pd.DataFrame(rows)
path = ROOT / "kinematic_features" / "output" / "kinematic_features.csv"
path.parent.mkdir(exist_ok=True)
out.to_csv(path, index=False)
print(f"{len(out)} recordings -> {path.relative_to(ROOT)} ({skipped} skipped as too short)")

2385 recordings -> kinematic_features/output/kinematic_features.csv (0 skipped as too short)


## 3. Summary

In [5]:
out.groupby(["source_dataset", "label"]).size().unstack(fill_value=0)

label,0,1
source_dataset,,
handydiag,630,622
newhandpd,420,372
pahaw_examples,65,64
uci358,10,0
uci395,37,165


Mean of key features by label (0 = HC, 1 = PD). Descriptive only: recordings are not independent.

In [6]:
out.groupby("label")[["velocity_mean", "pressure_mean", "in_air_time_s", "tremor_ratio"]].mean()

,velocity_mean,pressure_mean,in_air_time_s,tremor_ratio
label,,,,
0,2.077032,0.511178,0.174716,0.164685
1,1.795354,0.539398,0.388913,0.213150
